## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Google Generative AI encoder
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [8]:
import os

load_dotenv(override=True)

grok_base_url = os.getenv("GROK_BASE_URL")
grok_api_key = os.getenv("GROK_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

print("grok_base_url: ", grok_base_url)
print("grok_api_key: ", grok_api_key[:4])
print("gemini_api_key: ", gemini_api_key[:4])


grok_base_url:  https://api.x.ai/v1
grok_api_key:  xai-
gemini_api_key:  AIza


In [5]:
MODEL = "grok-4-1-fast-reasoning"
DB_NAME = "vector_db"

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=gemini_api_key)

# 1. Initialize an empty vectorstore
vectorstore = Chroma(
    embedding_function=embeddings, 
    persist_directory=DB_NAME
)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [9]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(api_key=grok_api_key, base_url=grok_base_url, temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [10]:
retriever.invoke("Who is Avery?")

[Document(id='32e4904d-d15a-4c2e-aed0-223f53b99676', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.'),
 Document(id='25d2c9dd-2cbd-4524-82ca-505b52a7307c', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial en

In [12]:
# It doesn't know anything about the retriever!!!!
# It doesn't currently now anything about Avery from the documents!
llm.invoke("Who is Avery?")

AIMessage(content='**Avery** is a common unisex given name of Old English origin, derived from the surname meaning "ruler of the elves" (from ælf "elf" + rīce "ruler"). \n\nIt could refer to many notable people, depending on context. Here are some prominent ones:\n\n### Entertainment & Media\n- **Avery Brooks** (b. 1948): American actor, director, and singer, best known as Commander Benjamin Sisko in *Star Trek: Deep Space Nine*.\n- **Jackson Avery**: Fictional character played by Jesse Williams on the TV series *Grey\'s Anatomy* (Shondaland universe).\n- **Avery** (from Disney\'s *DuckTales* reboot): A teenage girl and one of the main protagonists alongside her sister Lexi.\n\n### Sports\n- **Avery Johnson** (b. 1965): Former NBA point guard, coach, and current college basketball analyst; nicknamed "The Little General."\n- **Avery Bradley** (b. 1990): Retired NBA player known for his defense with teams like the Boston Celtics and LA Clippers.\n\n### Other\n- **Avery Dulles** (1918–200

## Time to put this together!

In [17]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.  Do not make things up!
If the question is not about Insurellm, say you are an Insurellm assistant and you don't know the answer.
Context:
{context}
"""

In [18]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [19]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster (likely who you mean, as "Averi" may be a slight misspelling) is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. Born on March 15, 1985, she\'s based in San Francisco, California, and has led the company since co-founding it in 2015.\n\nPrior to Insurellm, she was a Senior Product Manager at Innovate Insurance Solutions from 2013 to 2015. Under her leadership, Insurellm has grown into a leading Insurance Tech provider, launching products like Markellm (insurance marketplace), Carllm (auto insurance), Homellm (home insurance), and Rellm (reinsurance platform). She\'s known for innovative strategies, risk management expertise, diversity initiatives, work-life balance improvements, and community outreach on financial literacy.\n\nIn recent performance reviews, she received "Satisfactory" in 2022 and "Exceeds Expectations" in 2023, helping regain market leadership. Her current salary is $225,000. Let me know if you\'d like more details about her role or Insu

## What could possibly come next? 😂

In [20]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!